# XSpace context：跨线程起点与完成点

本 notebook 复查已保存的 `RUN-CPU` 证据；本次执行是 `REPLAY-OFFLINE`。不加载 executable，不运行当前 kernel 中的 JAX 编译。原始采集绑定源码 wheel 003；当前 kernel 可使用宿主环境完成数据解析与 NumPy 检查。

先阅读 [机制与复跑命令](xspace-contexts.md)。原始大文件保存在本仓库忽略目录，缺失时应按说明重新采集，不能把 notebook 输出当作原始证据。

In [1]:
from pathlib import Path
import json, sys, gzip
import numpy as np
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "upstream-sources.lock").exists())
sys.path.insert(0, str(ROOT / "research/software-stack"))
from cpu_executable_parser import inventory, message
from xspace_contexts import load_schema, flatten, context_groups, unique_pairs, json_correspondence
from verify_research import read_json, local_path, sha256
result = read_json(ROOT / "research/software-stack/xspace-context-results.json")
capture = local_path(result["audit_manifest"]["path"]).parent
assert inventory(capture, "REPLAY-OFFLINE") == result["audit_manifest"]
for key in ("host_runtime", "thunk_runtime"):
    assert inventory(local_path(result[key]["path"]).parent, "RUN-CPU") == result[key]
pool = load_schema(capture)
print({"audit_artifacts": result["audit_manifest"]["artifact_count"], "host_artifacts": result["host_runtime"]["artifact_count"], "producer_build": result["build_id"]})

{'audit_artifacts': 45, 'host_artifacts': 17, 'producer_build': 'kickoff-cpu-source-003'}


## 从 protobuf 重建关联

关联键先使用原始文件 scope、PID、type 与精确 uint64 ID。JSON 中名称和时间只用于检查已选中的原始事件如何导出，不能反过来猜 context。

In [2]:
counts = []
for case in result["cases"]:
    path = local_path(case["xspace"]["path"])
    assert sha256(path) == case["xspace"]["sha256"]
    rows = flatten(message(pool, "tensorflow.profiler.XSpace", path.read_bytes()))
    groups = context_groups(rows, sha256(path))
    selected = (lambda r: r["name"] in {"research_link_send", "research_link_receive"}) if case["mode"] == "host" else None
    pairs = unique_pairs(rows, groups, selected)
    assert rows == read_json(capture / case["label"] / "rows.json")
    assert pairs == case["pairs"]
    with gzip.open(local_path(case["json"]["path"]), "rt") as stream:
        exported = json.load(stream)
    assert json_correspondence(rows, exported["traceEvents"], pairs) == case["json_matches"]
    counts.append((case["label"], len(pairs), sum(p["cross_thread"] for p in pairs)))
assert sum(c[1] for c in counts) == 49
assert sum(c[2] for c in counts) == 13
counts

[('matmul', 3, 0),
 ('vmap_matmul', 9, 3),
 ('grad_matmul', 15, 3),
 ('jit_grad_vmap_matmul', 18, 3),
 ('explicit_host', 4, 4)]

## 大整数与逆序完成

四个 work 在提交线程产生 send，在各工作线程产生 receive。计数与身份来自原始 context；producer 的局部 duration 不等于逻辑任务耗时。gate 故意改变完成次序，不能据这些时间分析性能。

In [3]:
host = next(c for c in result["cases"] if c["label"] == "explicit_host")
ops = host["explicit_operations"]
ids = [o["context_id"] for o in ops]
assert ids == [str(2**60), str(2**60 + 1), str(2**63 + 7), str(2**64 - 1)]
assert float(ids[0]) == float(ids[1]) and ids[0] != ids[1]
assert [o["work"] for o in sorted(ops, key=lambda o: o["consumer_ps"])] == [3, 2, 1, 0]
assert ops[3]["outcome"] == "expected-error"
zero_clamps = sum(m["raw_duration_ps"] == 0 and m["exported_duration_ps"] == 1 for c in result["cases"] for m in c["json_matches"])
assert zero_clamps == 18
print({"exact_context_ids": ids, "completion_order": [3, 2, 1, 0], "zero_duration_exported_as_1ps": zero_clamps})

{'exact_context_ids': ['1152921504606846976', '1152921504606846977', '9223372036854775815', '18446744073709551615'], 'completion_order': [3, 2, 1, 0], 'zero_duration_exported_as_1ps': 18}


## 独立数值参考

四个 `A[16,16] @ W[16,24]`；work 3 在计算 ready 后才抛出预期异常，因此仍必须具有正确输出。

In [4]:
host_capture = local_path(result["host_runtime"]["path"]).parent
errors = []
with np.load(host_capture / "inputs.npz", allow_pickle=False) as data, np.load(host_capture / "outputs.npz", allow_pickle=False) as outputs:
    for i in range(4):
        reference = data[f"a_{i}"].astype(np.float64) @ data["weight"].astype(np.float64)
        actual = outputs[f"output_{i}"]
        np.testing.assert_allclose(actual, reference, rtol=2e-5, atol=2e-5)
        errors.append(float(np.max(np.abs(actual-reference))))
assert errors == result["host_numerical_max_errors"]
errors

[1.4665195405272335e-08,
 9.886395146985194e-09,
 1.4221048172391448e-08,
 1.7346300035248063e-08]

## 拒绝错误关联

这些是归档解析的边界测试，不是额外设备运行。生成的 `derived-trace.json` 是本工具 overlay，尚未运行 XProf 原生 preprocessing 或验证 viewer 渲染；设备内部与 TPU LLO 仍未验收。

In [5]:
from verify_xspace_contexts import selftest
checks = selftest(capture)
assert checks["negative_tests"] == result["negative_tests"]
assert checks["positive_edge_cases"] == result["positive_edge_cases"]
print({"rejected_invalid_cases": len(checks["negative_tests"]), "positive_boundaries": len(checks["positive_edge_cases"]), "evidence_level": "REPLAY-OFFLINE"})

{'rejected_invalid_cases': 20, 'positive_boundaries': 6, 'evidence_level': 'REPLAY-OFFLINE'}
